# Inserting, Updating, and Deleting

*Change selected records with bound values and distinguish committing from rolling back a transaction.*


## Prepare the Sales Table

The sales table holds five prepared records in an in-memory
SQLite database. `sale_id` is a primary key, and the other fields store
the product, category, quantity, whole-dollar unit price, and month number.
`execute()` runs one SQL statement; `executemany()` binds each prepared
tuple to the `?` placeholders. `commit()` completes the insert transaction.


In [1]:
import sqlite3

connection = sqlite3.connect(":memory:")

connection.execute("""
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price INTEGER NOT NULL,
    month INTEGER NOT NULL
)
""")

sales_rows = [
    ("Mouse", "Accessory", 3, 25, 1),
    ("Keyboard", "Accessory", 2, 45, 1),
    ("Monitor", "Display", 1, 180, 2),
    ("Laptop Stand", "Accessory", 4, 30, 2),
    ("Tablet", "Device", 2, 220, 3),
]

connection.executemany("""
INSERT INTO sales (product_name, category, quantity, unit_price, month)
VALUES (?, ?, ?, ?, ?)
""", sales_rows)
connection.commit()

print("Prepared sales:")
for row in connection.execute("SELECT * FROM sales ORDER BY sale_id"):
    print(row)


Prepared sales:
(1, 'Mouse', 'Accessory', 3, 25, 1)
(2, 'Keyboard', 'Accessory', 2, 45, 1)
(3, 'Monitor', 'Display', 1, 180, 2)
(4, 'Laptop Stand', 'Accessory', 4, 30, 2)
(5, 'Tablet', 'Device', 2, 220, 3)


There are five sales, numbered 1 through 5. Mouse, Keyboard, and
Laptop Stand belong to Accessory; Monitor belongs to Display,
and Tablet belongs to Device.


## Insert a New Sale

`INSERT INTO` adds a record. Bind a tuple of values to the placeholders,
then call `commit()` to retain the change. The primary key is assigned
automatically because it is omitted from the column list.


In [2]:
new_sale = ("Webcam", "Accessory", 3, 70, 3)
connection.execute("""
INSERT INTO sales (product_name, category, quantity, unit_price, month)
VALUES (?, ?, ?, ?, ?)
""", new_sale)
connection.commit()

print("Rows after insert:", connection.execute("SELECT COUNT(*) FROM sales").fetchone()[0])
print("Inserted record:", connection.execute("SELECT * FROM sales WHERE sale_id = ?", (6,)).fetchone())


Rows after insert: 6
Inserted record: (6, 'Webcam', 'Accessory', 3, 70, 3)


The table now contains six rows. Webcam receives sale ID 6 and contributes 3 × 70 = 210 dollars of revenue.


## Update a Record by Its Primary Key

`UPDATE sales SET ... WHERE sale_id = ?` changes only the identified
record. Preview the target before updating. Omitting `WHERE` would affect
every row, so keep the intended condition visible.
`execute()` returns a cursor; its `.rowcount` reports how many rows this
update affected. `.fetchone()` reads one row from a SELECT result.


In [3]:
print("Keyboard before update:", connection.execute(
    "SELECT sale_id, product_name, unit_price FROM sales WHERE sale_id = ?", (2,)
).fetchone())

cursor = connection.execute(
    "UPDATE sales SET unit_price = ? WHERE sale_id = ?", (50, 2)
)
connection.commit()

print("Updated rows:", cursor.rowcount)
print("Keyboard after update:", connection.execute(
    "SELECT sale_id, product_name, unit_price FROM sales WHERE sale_id = ?", (2,)
).fetchone())


Keyboard before update: (2, 'Keyboard', 45)
Updated rows: 1
Keyboard after update: (2, 'Keyboard', 50)


One row changes: Keyboard's unit price becomes 50. Its quantity stays 2, so its revenue increases from 90 to 100 dollars.


## Delete a Selected Record

`DELETE FROM ... WHERE ...` removes matching rows, not the table's schema.
This example removes only the prepared Monitor record, sale ID 3.


In [4]:
print("Record to remove:", connection.execute(
    "SELECT sale_id, product_name FROM sales WHERE sale_id = ?", (3,)
).fetchone())
cursor = connection.execute("DELETE FROM sales WHERE sale_id = ?", (3,))
connection.commit()

print("Deleted rows:", cursor.rowcount)
print("Remaining sales:")
for row in connection.execute("""
SELECT sale_id, product_name, quantity, unit_price, month
FROM sales
ORDER BY month, sale_id
"""):
    print(row)
print("Remaining row count:", connection.execute("SELECT COUNT(*) FROM sales").fetchone()[0])


Record to remove: (3, 'Monitor')
Deleted rows: 1
Remaining sales:
(1, 'Mouse', 3, 25, 1)
(2, 'Keyboard', 2, 50, 1)
(4, 'Laptop Stand', 4, 30, 2)
(5, 'Tablet', 2, 220, 3)
(6, 'Webcam', 3, 70, 3)
Remaining row count: 5


Five sales remain. Their IDs are 1, 2, 4, 5, and 6; deleting a record does not renumber the surviving primary keys.


## Roll Back an Uncommitted Change

A **transaction** groups changes that can be committed or undone together.
With Python 3.12's default `sqlite3` transaction settings, an `INSERT`,
`UPDATE`, or `DELETE` begins a transaction if none is open. `commit()`
retains its changes, while `rollback()` discards its uncommitted changes.
The same connection can read a tentative value before it is committed.


In [5]:
connection.execute("UPDATE sales SET quantity = ? WHERE sale_id = ?", (99, 1))
print("Tentative Mouse quantity:", connection.execute(
    "SELECT quantity FROM sales WHERE sale_id = ?", (1,)
).fetchone()[0])

connection.rollback()
print("Mouse quantity after rollback:", connection.execute(
    "SELECT quantity FROM sales WHERE sale_id = ?", (1,)
).fetchone()[0])


Tentative Mouse quantity: 99
Mouse quantity after rollback: 3


The tentative quantity is 99, but rollback restores Mouse quantity 3. Earlier committed insert, update, and delete operations remain in effect.


## Close the Connection

`.close()` releases the database connection. This database lives only
in memory, so its tables disappear when the connection closes.


In [6]:
connection.close()
print("Connection closed.")


Connection closed.


Use primary keys and explicit conditions for targeted changes. Bind values and decide deliberately whether to commit or roll back the current transaction.

[sqlite3 transaction control](https://docs.python.org/3.12/library/sqlite3.html#transaction-control)
